# 26. Text-Image Baselines — Zero-Shot SigLIP2 & CLIP

**Zero-shot reference points** for the text-image retrieval experiments. No training, no projection heads, no fine-tuning — just the pretrained models evaluated directly on the same GT protocol as NB 28 / NB 29.

### Purpose

Every NB 02 / 09 / 27 / 28 result needs a baseline to be interpretable. This notebook produces those baselines for the two standard contrastive text-image models:

| Model | Context | Dim | Source |
|---|---|---|---|
| **SigLIP2** | 64 tokens | 768 | `google/siglip2-base-patch16-224` |
| **CLIP** | 77 tokens | 512 | `openai/clip-vit-base-patch32` |

### Alignment with NB 28 / 29

| Element | Match |
|---|---|
| HF cache (`cache_dir=HF_CACHE`) | ✓ `hf_cache2` |
| Logging (single shared log + `img_inbatch_ft` logger + stdout echo) | ✓ |
| GPU pinning (`cuda:0`) | ✓ |
| GT set (500 records, `gt_textimage_records.pkl`) | ✓ |
| `compute_retrieval_metrics` (identical function) | ✓ |
| `gt_results.csv` per model + combined `summary_all_models.csv` | ✓ |
| **Truncation to native max-length** (not chunking) | Zero-shot standard |

### ⚠️ Honest limitation — truncation loses most of the text

BSV texts average **hundreds** of tokens, far beyond the native context of either model (64 / 77 tokens). True zero-shot truncates to native context, meaning SigLIP2 sees ~10% of each passage and CLIP sees ~5%. **These numbers are the literature-standard zero-shot baseline** — comparable to every published CLIP/SigLIP zero-shot result — but they understate what either model would achieve on the full text.

For a task-honest evaluation of a *chunked + mean-pooled* variant, see NB 09's methodology. That's a **different experiment** (the model wasn't trained for chunked input), so it doesn't belong in the zero-shot baseline. If you want that comparison, add a `meanpool` eval mode to this notebook and label those rows separately.

### Output layout

```
OUTPUT_DIR = results/runs/section_4/baselines/zeroshot_textimage/
  zeroshot_baselines_ft.log
  siglip2_truncate64/gt_results.csv
  clip_truncate77/gt_results.csv
  summary_all_models.csv
```

### Skip-if-done

Each model produces `gt_results.csv` on first run; subsequent runs skip. Delete the per-model directory to re-evaluate.

In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


## Section 0 — Installations

Same pin set as NB 28 / 29.

In [ ]:
# !pip install openpyxl
# !pip install pandas --upgrade
# !pip install accelerate
# !pip install transformers==4.57.6
# !pip install --upgrade huggingface_hub

### Both models are ungated — no HuggingFace login required.

## Section 1 — Setup, logging, config

Log file inside `OUTPUT_DIR`, GPU pinned to `cuda:0` (matches NB 28).

In [1]:
import os
import sys
import time
import math
import pickle
import logging
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModel, AutoProcessor, CLIPModel, CLIPProcessor
import transformers

In [2]:
transformers.logging.set_verbosity_error()

# --- Paths: define BOTH root and baseline dir up front so log can live inside ---
OUTPUT_ROOT = str(RESULTS_RUNS_ROOT)
OUTPUT_DIR  = os.path.join(OUTPUT_ROOT, "baselines", "zeroshot_textimage")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)

LOG_FORMAT = '%(asctime)s : %(levelname)s : %(message)s'
LOG_FILE   = os.path.join(OUTPUT_DIR, "zeroshot_baselines_ft.log")

logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format=LOG_FORMAT)
logger = logging.getLogger("img_inbatch_ft")
if not any(type(h) is logging.StreamHandler for h in logger.handlers):
    _sh = logging.StreamHandler(sys.stdout)
    _sh.setFormatter(logging.Formatter(LOG_FORMAT))
    logger.addHandler(_sh)


# --- GPU pinning: always cuda:0 (matches NB 28) ---
CUDA = "cuda:0"
device = CUDA if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)
logger.info("Device: %s", device)
if device == CUDA:
    logger.info("GPU: %s (physical index %d of %d visible)",
                torch.cuda.get_device_name(torch.cuda.current_device()),
                torch.cuda.current_device(),
                torch.cuda.device_count())
logger.info("LOG_FILE = %s", LOG_FILE)
logger.info("OUTPUT_DIR = %s", OUTPUT_DIR)

2026-10-22 23:15:06,727 : INFO : Device: cuda:0
2026-10-22 23:15:06,735 : INFO : GPU: Quadro RTX 8000 (physical index 0 of 2 visible)
2026-10-22 23:15:06,744 : INFO : LOG_FILE = /projects/agrifutur/sji/cross_encoder_ft/baselines/zeroshot_textimage/zeroshot_baselines_ft.log
2026-10-22 23:15:06,745 : INFO : OUTPUT_DIR = /projects/agrifutur/sji/cross_encoder_ft/baselines/zeroshot_textimage


### Config

In [10]:
HF_CACHE = DEFAULT_HF_CACHE
VARS_DIR = str(FINETUNING_ROOT / "vars")
os.makedirs(VARS_DIR, exist_ok=True)

# --- Models ---
SIGLIP2_MODEL_NAME = "google/siglip2-base-patch16-224"
CLIP_MODEL_NAME    = "openai/clip-vit-base-patch32"

# --- Native max-lengths (truncation targets) ---
SIGLIP2_MAX_LEN = 64
CLIP_MAX_LEN    = 77

# --- Batching ---
IMAGE_BATCH_SIZE = 16
TEXT_BATCH_SIZE  = 32

# --- Metrics ---
K_VALUES = [1, 3, 5, 10]

# --- Model keys (also the per-model subdirectory names) ---
MODEL_KEYS = [
    "siglip2_truncate64",
    "siglip2_meanpool64",    
    "clip_truncate77",
    "clip_meanpool77",       
]

logger.info("Config loaded. SIGLIP2_MAX_LEN=%d, CLIP_MAX_LEN=%d", SIGLIP2_MAX_LEN, CLIP_MAX_LEN)
logger.info("MODEL_KEYS=%s", MODEL_KEYS)

2026-10-22 23:31:31,834 : INFO : Config loaded. SIGLIP2_MAX_LEN=64, CLIP_MAX_LEN=77
2026-10-22 23:31:31,836 : INFO : MODEL_KEYS=['siglip2_truncate64', 'siglip2_meanpool64', 'clip_truncate77', 'clip_meanpool77']


## Section 2 — Load GT set

Reuses `vars/gt_textimage_records.pkl` produced by NB 28's Section 6 — same 500-record set NB 28 evaluates on, so zero-shot numbers are directly comparable to NB 28's trained numbers.

In [11]:
def reload_vars(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

GT_RECORDS_PATH = os.path.join(VARS_DIR, "gt_textimage_records.pkl")
if not os.path.exists(GT_RECORDS_PATH):
    raise FileNotFoundError(
        f"Missing {GT_RECORDS_PATH}.\n"
        "Run NB 28's Section 6 once to build it, then re-run this cell."
    )

gt_textimage_records = reload_vars(GT_RECORDS_PATH)
logger.info("Loaded %d GT text/image records", len(gt_textimage_records))
assert len(gt_textimage_records) > 0

_s = gt_textimage_records[0]
for _k in ("text", "image_path"):
    assert _k in _s, f"missing key {_k!r} in gt_textimage_records"
logger.info("GT schema OK")

2026-10-22 23:31:56,087 : INFO : Loaded 500 GT text/image records
2026-10-22 23:31:56,089 : INFO : GT schema OK


## Section 3 — Metrics + shared zero-shot helpers

`compute_retrieval_metrics` is **identical** to NB 28's — same ranking rule, same `R@k` / `MRR` computation.

In [12]:
def _chunk_text_for_tokenizer(text, tokenizer, chunk_size):
    """Split text into chunks of `chunk_size` content tokens, decoding each back to string.
    Matches NB 09 / 27D's chunking convention (chunk on raw token ids, re-encode via processor)."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    if not ids:
        return [text]
    chunks = [ids[i:i + chunk_size] for i in range(0, len(ids), chunk_size)]
    return [tokenizer.decode(c) for c in chunks]

In [13]:
def compute_retrieval_metrics(sim_matrix, k_values=K_VALUES):
    N = sim_matrix.shape[0]
    true_scores = sim_matrix.diag().unsqueeze(1)
    ranks = (sim_matrix >= true_scores).sum(dim=1)
    results = {}
    for k in k_values:
        results[f"R@{k}"] = (ranks <= k).float().mean().item()
    results["MRR"] = (1.0 / ranks.float()).mean().item()
    results["N"] = N
    return results


def _extract_feat(out):
    """Handle both return types across transformers versions:
    BaseModelOutputWithPooling (use .pooler_output) or raw Tensor (use directly)."""
    feat = out.pooler_output if hasattr(out, "pooler_output") else out
    return F.normalize(feat, p=2, dim=-1)


logger.info("Metrics + helpers ready.")

2026-10-22 23:32:08,973 : INFO : Metrics + helpers ready.


## Section 4 — SigLIP2 zero-shot (truncate@64)

Encodes all 500 GT images and 500 GT texts with the pretrained SigLIP2 towers, L2-normalizes, computes cosine sim, runs the same metrics as NB 28.

No projection heads — the raw `get_image_features` / `get_text_features` outputs are used.

In [6]:
def run_siglip2_zeroshot(gt_records):
    processor = AutoProcessor.from_pretrained(SIGLIP2_MODEL_NAME, cache_dir=HF_CACHE)
    model = AutoModel.from_pretrained(SIGLIP2_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    model.eval()

    def encode_images(paths):
        chunks = []
        with torch.no_grad():
            for i in range(0, len(paths), IMAGE_BATCH_SIZE):
                imgs = [Image.open(p).convert("RGB") for p in paths[i:i + IMAGE_BATCH_SIZE]]
                inputs = processor(images=imgs, return_tensors="pt").to(device)
                feat = _extract_feat(model.get_image_features(**inputs))
                chunks.append(feat.cpu())
        return torch.cat(chunks, dim=0)

    def encode_texts(texts):
        chunks = []
        with torch.no_grad():
            for i in range(0, len(texts), TEXT_BATCH_SIZE):
                inputs = processor(
                    text=texts[i:i + TEXT_BATCH_SIZE],
                    padding="max_length", max_length=SIGLIP2_MAX_LEN,
                    truncation=True, return_tensors="pt",
                ).to(device)
                feat = _extract_feat(model.get_text_features(**inputs))
                chunks.append(feat.cpu())
        return torch.cat(chunks, dim=0)

    t0 = time.time()
    img_emb = encode_images([r["image_path"] for r in gt_records])
    txt_emb = encode_texts([r["text"] for r in gt_records])
    logger.info("[SigLIP2] Encoded %d images + %d texts in %.1fs (dim=%d)",
                len(img_emb), len(txt_emb), time.time() - t0, img_emb.shape[-1])

    sim = img_emb @ txt_emb.T
    summary = pd.DataFrame([
        {"direction": "text -> image", **compute_retrieval_metrics(sim.T)},
        {"direction": "image -> text", **compute_retrieval_metrics(sim)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]

    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

logger.info("run_siglip2_zeroshot() ready.")

2026-10-22 23:15:06,906 : INFO : run_siglip2_zeroshot() ready.


## Section 4B — SigLIP2 zero-shot mean pool (truncate@64)

In [14]:
def run_siglip2_meanpool(gt_records):
    """SigLIP2 zero-shot with mean-pooled chunk text embeddings (chunk size = 64)."""
    processor = AutoProcessor.from_pretrained(SIGLIP2_MODEL_NAME, cache_dir=HF_CACHE)
    model = AutoModel.from_pretrained(SIGLIP2_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    model.eval()
    tokenizer = processor.tokenizer

    def encode_images(paths):
        chunks = []
        with torch.no_grad():
            for i in range(0, len(paths), IMAGE_BATCH_SIZE):
                imgs = [Image.open(p).convert("RGB") for p in paths[i:i + IMAGE_BATCH_SIZE]]
                inputs = processor(images=imgs, return_tensors="pt").to(device)
                feat = _extract_feat(model.get_image_features(**inputs))
                chunks.append(feat.cpu())
        return torch.cat(chunks, dim=0)

    def encode_texts_meanpool(texts):
        # Split all texts into chunks, remember per-sample chunk counts
        all_chunks, chunk_counts = [], []
        for t in texts:
            cs = _chunk_text_for_tokenizer(t, tokenizer, SIGLIP2_MAX_LEN)
            all_chunks.extend(cs)
            chunk_counts.append(len(cs))

        # Encode all chunks in batches
        all_feats = []
        with torch.no_grad():
            for i in range(0, len(all_chunks), TEXT_BATCH_SIZE):
                inputs = processor(
                    text=all_chunks[i:i + TEXT_BATCH_SIZE],
                    padding="max_length", max_length=SIGLIP2_MAX_LEN,
                    truncation=True, return_tensors="pt",
                ).to(device)
                all_feats.append(_extract_feat(model.get_text_features(**inputs)).cpu())
        all_feats = torch.cat(all_feats, dim=0)

        # Mean-pool per-sample chunks, re-normalize
        pooled, idx = [], 0
        for n in chunk_counts:
            sample = all_feats[idx:idx + n]
            m = sample.mean(dim=0)
            m = F.normalize(m, p=2, dim=0)
            pooled.append(m)
            idx += n
        return torch.stack(pooled)

    t0 = time.time()
    img_emb = encode_images([r["image_path"] for r in gt_records])
    txt_emb = encode_texts_meanpool([r["text"] for r in gt_records])
    logger.info("[SigLIP2-meanpool] Encoded %d images + %d texts in %.1fs (dim=%d)",
                len(img_emb), len(txt_emb), time.time() - t0, img_emb.shape[-1])

    sim = img_emb @ txt_emb.T
    summary = pd.DataFrame([
        {"direction": "text -> image", **compute_retrieval_metrics(sim.T)},
        {"direction": "image -> text", **compute_retrieval_metrics(sim)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]

    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

logger.info("run_siglip2_meanpool() ready.")

2026-10-22 23:33:38,000 : INFO : run_siglip2_meanpool() ready.


## Section 5 — CLIP zero-shot (truncate@77)

Same protocol as SigLIP2, using `CLIPModel` + `CLIPProcessor`.

In [7]:
def run_clip_zeroshot(gt_records):
    processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME, cache_dir=HF_CACHE)
    model = CLIPModel.from_pretrained(CLIP_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    model.eval()

    def encode_images(paths):
        chunks = []
        with torch.no_grad():
            for i in range(0, len(paths), IMAGE_BATCH_SIZE):
                imgs = [Image.open(p).convert("RGB") for p in paths[i:i + IMAGE_BATCH_SIZE]]
                inputs = processor(images=imgs, return_tensors="pt").to(device)
                feat = _extract_feat(model.get_image_features(**inputs))
                chunks.append(feat.cpu())
        return torch.cat(chunks, dim=0)

    def encode_texts(texts):
        chunks = []
        with torch.no_grad():
            for i in range(0, len(texts), TEXT_BATCH_SIZE):
                inputs = processor(
                    text=texts[i:i + TEXT_BATCH_SIZE],
                    padding="max_length", max_length=CLIP_MAX_LEN,
                    truncation=True, return_tensors="pt",
                ).to(device)
                feat = _extract_feat(model.get_text_features(**inputs))
                chunks.append(feat.cpu())
        return torch.cat(chunks, dim=0)

    t0 = time.time()
    img_emb = encode_images([r["image_path"] for r in gt_records])
    txt_emb = encode_texts([r["text"] for r in gt_records])
    logger.info("[CLIP] Encoded %d images + %d texts in %.1fs (dim=%d)",
                len(img_emb), len(txt_emb), time.time() - t0, img_emb.shape[-1])

    sim = img_emb @ txt_emb.T
    summary = pd.DataFrame([
        {"direction": "text -> image", **compute_retrieval_metrics(sim.T)},
        {"direction": "image -> text", **compute_retrieval_metrics(sim)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]

    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

logger.info("run_clip_zeroshot() ready.")

2026-10-22 23:15:06,935 : INFO : run_clip_zeroshot() ready.


## Section 5B — CLIP zero-shot mean pool (truncate@64)

In [15]:
def run_clip_meanpool(gt_records):
    """CLIP zero-shot with mean-pooled chunk text embeddings (chunk size = 77)."""
    processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME, cache_dir=HF_CACHE)
    model = CLIPModel.from_pretrained(CLIP_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    model.eval()
    tokenizer = processor.tokenizer

    def encode_images(paths):
        chunks = []
        with torch.no_grad():
            for i in range(0, len(paths), IMAGE_BATCH_SIZE):
                imgs = [Image.open(p).convert("RGB") for p in paths[i:i + IMAGE_BATCH_SIZE]]
                inputs = processor(images=imgs, return_tensors="pt").to(device)
                feat = _extract_feat(model.get_image_features(**inputs))
                chunks.append(feat.cpu())
        return torch.cat(chunks, dim=0)

    def encode_texts_meanpool(texts):
        all_chunks, chunk_counts = [], []
        for t in texts:
            cs = _chunk_text_for_tokenizer(t, tokenizer, CLIP_MAX_LEN)
            all_chunks.extend(cs)
            chunk_counts.append(len(cs))

        all_feats = []
        with torch.no_grad():
            for i in range(0, len(all_chunks), TEXT_BATCH_SIZE):
                inputs = processor(
                    text=all_chunks[i:i + TEXT_BATCH_SIZE],
                    padding="max_length", max_length=CLIP_MAX_LEN,
                    truncation=True, return_tensors="pt",
                ).to(device)
                all_feats.append(_extract_feat(model.get_text_features(**inputs)).cpu())
        all_feats = torch.cat(all_feats, dim=0)

        pooled, idx = [], 0
        for n in chunk_counts:
            sample = all_feats[idx:idx + n]
            m = sample.mean(dim=0)
            m = F.normalize(m, p=2, dim=0)
            pooled.append(m)
            idx += n
        return torch.stack(pooled)

    t0 = time.time()
    img_emb = encode_images([r["image_path"] for r in gt_records])
    txt_emb = encode_texts_meanpool([r["text"] for r in gt_records])
    logger.info("[CLIP-meanpool] Encoded %d images + %d texts in %.1fs (dim=%d)",
                len(img_emb), len(txt_emb), time.time() - t0, img_emb.shape[-1])

    sim = img_emb @ txt_emb.T
    summary = pd.DataFrame([
        {"direction": "text -> image", **compute_retrieval_metrics(sim.T)},
        {"direction": "image -> text", **compute_retrieval_metrics(sim)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]

    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

logger.info("run_clip_meanpool() ready.")

2026-10-22 23:34:11,963 : INFO : run_clip_meanpool() ready.


## Section 6 — Run both baselines

In [16]:
_RUNNERS = {
    "siglip2_truncate64": run_siglip2_zeroshot,
    "siglip2_meanpool64": run_siglip2_meanpool,     
    "clip_truncate77":    run_clip_zeroshot,
    "clip_meanpool77":    run_clip_meanpool,        
}

for key in MODEL_KEYS:
    model_dir = os.path.join(OUTPUT_DIR, key)
    os.makedirs(model_dir, exist_ok=True)
    gt_path = os.path.join(model_dir, "gt_results.csv")

    if os.path.exists(gt_path):
        logger.info("=== [%s] already completed, skipping ===", key)
        continue

    logger.info("=" * 70)
    logger.info("=== Zero-shot evaluation: %s ===", key)
    logger.info("=" * 70)

    summary = _RUNNERS[key](gt_textimage_records)
    summary.insert(0, "model", key)
    summary.to_csv(gt_path, index=False)
    logger.info("[%s] Results:\n%s", key, summary.to_string(index=False))

logger.info("All zero-shot baselines complete.")

2026-10-22 23:34:30,611 : INFO : === [siglip2_truncate64] already completed, skipping ===
2026-10-22 23:34:30,617 : INFO : ======================================================================
2026-10-22 23:34:30,618 : INFO : === Zero-shot evaluation: siglip2_meanpool64 ===
2026-10-22 23:34:30,620 : INFO : ======================================================================
2026-10-22 23:34:49,489 : INFO : [SigLIP2-meanpool] Encoded 500 images + 500 texts in 12.5s (dim=768)
2026-10-22 23:34:49,931 : INFO : [siglip2_meanpool64] Results:
             model     direction   N   R@1   R@3   R@5  R@10      MRR
siglip2_meanpool64 text -> image 500 0.018 0.040 0.048 0.098 0.048810
siglip2_meanpool64 image -> text 500 0.026 0.054 0.078 0.116 0.061545
2026-10-22 23:34:49,937 : INFO : === [clip_truncate77] already completed, skipping ===
2026-10-22 23:34:49,943 : INFO : ======================================================================
2026-10-22 23:34:49,949 : INFO : === Zero-shot evaluat

## Section 7 — Combined summary

In [17]:
rows = []
for key in MODEL_KEYS:
    p = os.path.join(OUTPUT_DIR, key, "gt_results.csv")
    if os.path.exists(p):
        rows.append(pd.read_csv(p))
    else:
        logger.warning("[%s] no gt_results.csv found, skipping", key)

if rows:
    summary_all_models = (pd.concat(rows, ignore_index=True)
                            .sort_values(["model", "direction"])
                            .reset_index(drop=True))
    summary_all_models.to_csv(os.path.join(OUTPUT_DIR, "summary_all_models.csv"), index=False)
    logger.info("=== Combined zero-shot baseline summary ===\n%s",
                summary_all_models.to_string(index=False))
    try:
        display(summary_all_models)
    except NameError:
        print(summary_all_models)
else:
    logger.warning("No gt_results.csv files found; nothing to summarize.")

2026-10-22 23:35:02,229 : INFO : === Combined zero-shot baseline summary ===
             model     direction   N   R@1   R@3   R@5  R@10      MRR
   clip_meanpool77 image -> text 500 0.014 0.040 0.056 0.084 0.042352
   clip_meanpool77 text -> image 500 0.008 0.026 0.046 0.076 0.035105
   clip_truncate77 image -> text 500 0.016 0.030 0.048 0.088 0.042886
   clip_truncate77 text -> image 500 0.012 0.032 0.044 0.072 0.038161
siglip2_meanpool64 image -> text 500 0.026 0.054 0.078 0.116 0.061545
siglip2_meanpool64 text -> image 500 0.018 0.040 0.048 0.098 0.048810
siglip2_truncate64 image -> text 500 0.024 0.054 0.070 0.108 0.056622
siglip2_truncate64 text -> image 500 0.018 0.050 0.058 0.098 0.049558


,model,direction,N,R@1,R@3,R@5,R@10,MRR
0,clip_meanpool77,image -> text,500,0.014,0.040,0.056,0.084,0.042352
1,clip_meanpool77,text -> image,500,0.008,0.026,0.046,0.076,0.035105
2,clip_truncate77,image -> text,500,0.016,0.030,0.048,0.088,0.042886
3,clip_truncate77,text -> image,500,0.012,0.032,0.044,0.072,0.038161
4,siglip2_meanpool64,image -> text,500,0.026,0.054,0.078,0.116,0.061545
5,siglip2_meanpool64,text -> image,500,0.018,0.040,0.048,0.098,0.048810
6,siglip2_truncate64,image -> text,500,0.024,0.054,0.070,0.108,0.056622
7,siglip2_truncate64,text -> image,500,0.018,0.050,0.058,0.098,0.049558
